# Data Cleaning Notebook for Job Data Analysis

This notebook handles the cleaning and preprocessing of raw job data for analysis. It focuses on skills enhancement, date conversion, salary cleaning, and location data standardization.

## 1. Project Setup and Configuration

Configure paths and import necessary libraries and modules.

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

# Add src to path for custom modules
sys.path.insert(0, '..')
print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Import Cleaning Modules

Import custom data cleaning modules for job data processing.

In [2]:
# Import the cleaning module
try:
    from src.cleaning.sql_cleaner import SQLCleaner 
    from src.cleaning.clean_job_title import HybridJobTitleClassifier
    from src.cleaning.clean_jobs import clean_job_type
    print("Cleaning modules imported successfully!")
except ImportError as e:
    print(f"Error importing cleaning modules: {e}")

PyTorch version 2.4.1 available.
Polars version 1.38.1 available.


Cleaning modules imported successfully!


## 3. Initialize Database Components
Set up the database connection and cleaning orchestrator.

In [3]:
# Initialize the SQL-based cleaner
sql_cleaner = SQLCleaner()

# Test database connection
if sql_cleaner.db.test_connection():
    print("Database connection successful")
else:
    print("Database connection failed - check your .env file")
    sys.exit(1)

Database connection test successful
Database connection successful


## 4. Database Setup
Run this only once to create tables and load mapping data. This is safe - it never drops existing data.

In [4]:
sql_cleaner.setup_database()

Setting up database tables and mappings...
Tables ready (from your SQL definition)
Database setup complete


## 5. Load Raw Data from Database
Examine the raw data structure before cleaning.

In [5]:
# Load raw data from database
with sql_cleaner.db.cursor(dict_mode=True) as cur:
    cur.execute("""
        SELECT title, skills, country,salary_min, salary_period, salary_currency, published
        FROM jobs_cleaned WHERE salary_min > 0
        ORDER BY title
        LIMIT 5
    """)
    raw_sample = cur.fetchall()
    
    cur.execute("SELECT COUNT(*) as count FROM jobs_cleaned")
    total_raw = cur.fetchone()['count']

raw_df = pd.DataFrame(raw_sample)
print(f"Total raw records in database: {total_raw}")
print(f"\nSample of {len(raw_df)} records:")
display(raw_df.head(3))

print("\nMissing values in raw data (sample):")
missing_raw = raw_df.isnull().sum()
for col, count in missing_raw.items():
    if count > 0:
        pct = (count / len(raw_df)) * 100
        print(f"   {col}: {count} ({pct:.1f}%)")

Total raw records in database: 20815

Sample of 5 records:


,title,skills,country,salary_min,salary_period,salary_currency,published
0,AI Engineer H/F,"[Machine Learning, Testing, Python, SQL, LLMs,...",France,45000,annual,EUR,2025-11-02 23:00:55.724781+01:00
1,AI Scientist,"[Machine Learning, NLP, LLMs, Data Science, Ap...",USA,90000,annual,USD,2025-06-13 13:06:00.690000+01:00
2,Data Scientist II,"[Python, SQL, R, Data Science, Data Analysis, ...",USA,40,hourly,USD,2025-11-16 23:01:25.760962+01:00



Missing values in raw data (sample):


## 6. Run SQL-Based Cleaning Steps
These operations use your SQL mapping tables for fast, set-based cleaning. They populate the `cleaned_jobs` table while preserving raw data.

In [6]:
sql_cleaner.run_sql_cleaning()

 Running SQL cleaning pipeline...
✓ 01 location_cleaning.sql
✓ 02 remove_duplicates.sql
✓ 03 remove_generic_skills.sql
✓ 04 date_extraction.sql
✓ 05 convert_salaries_to_usd.sql
SQL cleaning complete


In [ ]:
# Load raw data from database
with sql_cleaner.db.cursor(dict_mode=True) as cur:
    cur.execute("""
        SELECT *
        FROM jobs_cleaned
        
    """)
    raw_sample = cur.fetchall()

raw_df = pd.DataFrame(raw_sample)
display(raw_df.head(3), len(raw_df))


,job_slug,title,skills,seniority,types,country,city,salary_min,salary_max,salary_currency,salary_period,published,company_name,company_sector,country_original,salary_min_annual_usd,salary_max_annual_usd,salary_annual_usd
0,inspiren-senior-data-platform-engineer-eeq1,Senior Data Platform Engineer,"[AWS, Kafka, Data Infrastructure, Data Governa...",,"[Remote, Full Time]",Us:Canada,,800000,200000,USD,annual,2025_12,Inspiren,,US:Canada,800000.00,200000.00,None
1,stensul-data-analyst-zogc,Data Analyst,"[SQL, Agile, SAAS, Mathematics, JSON, Looker]",,[Full Time],United States,"New York, NY",115000,135000,USD,annual,2025_12,Stensul,,USA,115000.00,135000.00,None
2,spoton-corporate-senior-data-analyst-7247,Senior Data Analyst,"[Python, SQL, R, Data Infrastructure, SAAS, Tr...",,[Full Time],Poland,Krakow,11000,15500,PLN,annual,2025_09,SpotOn: Corporate,,Poland,2750.00,3875.00,None


6130

## 7. Job Title Classification with ML
This section uses a sentence transformer model to classify job titles into standardized roles and extract seniority levels.

### 7.1 Initialize classifier

In [8]:
classifier = HybridJobTitleClassifier() 
 
print(f"Categories: {len(classifier.categories)} job roles")

Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Computing category embeddings...
Categories: 12 job roles


### 7.2  Classify job titles
The model will:
1. Assign a standardized role category
2. Extract seniority level
3. Provide a similarity score

In [9]:
with sql_cleaner.db.cursor(dict_mode=True) as cur:
    cur.execute("""
        SELECT *
        FROM jobs_cleaned
    """)
    data = cur.fetchall()

df = pd.DataFrame(data)

df = classifier.classify_dataframe(df, title_col='title')
 

### 7.3 Update database with classification results
Write the standardized titles and seniority levels back to the `cleaned_jobs` table.

In [10]:
# Add columns and update classifications
with sql_cleaner.db.cursor(dict_mode=True) as cursor:
    cursor.execute("""
        ALTER TABLE jobs_cleaned 
        ADD COLUMN IF NOT EXISTS standardized_title VARCHAR(100),
        ADD COLUMN IF NOT EXISTS seniority_level VARCHAR(50),
        ADD COLUMN IF NOT EXISTS similarity_score FLOAT;
    """)
    
    # Update with classifications
    updated = 0
    for _, row in df.iterrows():
        cursor.execute("""
            UPDATE jobs_cleaned 
            SET standardized_title = %s,
                seniority_level = %s,
                similarity_score = %s
            WHERE job_slug = %s
        """, (
            row['cleaned_title'],
            row['seniority_level'],
            row['similarity_score'],
            row['job_slug']
        ))
        updated += cursor.rowcount

print(f"Added columns and updated {updated} records")

Added columns and updated 15099 records


## 8. Apply cleaning function
The function will:
1. Parse multilingual job type strings
2. Extract standardized job_type (full_time, part_time, contract)
3. Extract work_mode (remote, hybrid, onsite) 

In [11]:
# Apply cleaning function
df = clean_job_type(df)

print("Job type cleaning complete")
print(f"\nProcessed {len(df)} records")

Job types cleaned. 15099 records processed.


Job type cleaning complete

Processed 15099 records


### 8.1 Update database with cleaned values
Write the standardized job types back to the database.

In [15]:
# Update database with cleaned values
updated = 0
with sql_cleaner.db.cursor(dict_mode=True) as cursor:
    cursor.execute("""
        ALTER TABLE jobs_cleaned 
        ADD COLUMN IF NOT EXISTS job_type VARCHAR(100),
        ADD COLUMN IF NOT EXISTS work_mode VARCHAR(50);
    """)

    for _, row in df.iterrows():
        cursor.execute("""
            UPDATE jobs_cleaned 
            SET job_type = %s,
                work_mode = %s 
            WHERE job_slug = %s
        """, (
            row['job_type'] if pd.notna(row['job_type']) else None,
            row['work_mode'] if pd.notna(row['work_mode']) else None, 
            row['job_slug']
        ))
        updated += cursor.rowcount

print(f"Updated {updated} records in jobs_cleaned table")

Updated 15099 records in jobs_cleaned table


## 9. Verify the updates
Check that the database was updated correctly.

In [17]:
# Verify updates
display(df.head(3), len(df))

,job_slug,title,skills,seniority,types,country,city,salary_min,salary_max,salary_currency,...,company_sector,country_original,salary_min_annual_usd,salary_max_annual_usd,salary_annual_usd,cleaned_title,seniority_level,similarity_score,job_type,work_mode
0,rca-data-analyst-h-f-gl3q,Data Analyst H/F,"[Cloud, SAAS, Transformation]",,[Temporary],France,,0,0,,...,,France,0.00,0.00,None,Data Analyst,Unspecified,0.746577,contract,NaN
1,smartsheet-sr-machine-learning-operations-engi...,Sr. Machine Learning Operations Engineer,"[Python, LLMs, Tensorflow, PyTorch, Generative...",,"[Remote, Full Time]",Us:India,,0,0,,...,,US:India,0.00,0.00,None,Machine Learning Engineer,Senior,0.777551,NaN,remote
2,miratech-senior-data-analytics-cgxi,Senior Data Analytics,"[SQL, Tableau, UX, Flow]",,[Full Time],United States,,0,0,,...,,USA,0.00,0.00,None,Analytics Engineer,Senior,0.550527,full_time,NaN


15099